# Step 15 — Train at 224px on the laterality-fixed cache

**Settings: `GPU T4 x2`, `Internet On`.**

**Inputs:** competition data, `rsna-knee-13a-cache224lat-sagcor`, `rsna-knee-13b-cache224lat-axial`, `rsna-knee-03-extract-all`

## Two changes at once — but separately attributable

| | before | now |
|---|---|---|
| resolution | 160 x 160 | **224 x 224** |
| slices per series | 24 | **32** |
| laterality | resolved for 52.5% of studies | **100%** |

Normally you'd change one variable at a time. Here you don't have to, because the two effects show up in different places:

- **The laterality fix** should close the medial/lateral gaps: Meniscus 0.766 vs 0.668, OA 0.837 vs 0.745. If those converge, the mirroring bug is fixed.
- **Resolution** should lift everything, especially thin structures.

So one run answers both questions.

## This trains fold 0 only

~4 hours. Verify the gaps close and the score improves before committing 16 more hours to folds 1-4.

**Reference to beat:** 160px 5-fold ensemble scored gold 0.825, public LB **0.856**. Single fold 0 at 160px was gold 0.803.

## Memory

The 2.5D channel stack is built on the GPU inside `forward` rather than in the dataset, so the DataLoader moves 3x less data. Batch of 3 studies = 288 slices per step, with gradient accumulation to keep the effective batch at 6, matching the 160px runs.

### 1 · Setup — two caches concatenated

In [ ]:
import os, gc, glob, time, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
A_PATH = glob.glob("/kaggle/input/notebooks/*/*/cache224lat_a.npy")[0]
B_PATH = glob.glob("/kaggle/input/notebooks/*/*/cache224lat_b.npy")[0]
PSEUDO = glob.glob("/kaggle/input/notebooks/*/*/pseudo_labels.csv")[0]
GOLD   = glob.glob("/kaggle/input/notebooks/*/*/gold58.csv")[0]

arrA = np.load(A_PATH, mmap_mode="r")   # (N, 2, 32, 224, 224) sagittal, coronal
arrB = np.load(B_PATH, mmap_mode="r")   # (N, 1, 32, 224, 224) axial
print("A", arrA.shape, "\nB", arrB.shape)

N, _, NSL, PX, _ = arrA.shape
NSER = arrA.shape[1] + arrB.shape[1]

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]

train  = pd.read_csv(f"{ROOT}/train.csv")
pseudo = pd.read_csv(PSEUDO)
gold   = pd.read_csv(GOLD)
assert (pseudo["StudyInstanceUID"].values == train["StudyInstanceUID"].values).all()

Y = pseudo[KEYS].values.astype(np.float32) / 100.0
nan_mask = np.isnan(Y).any(1)
Y = np.nan_to_num(Y, nan=0.2).astype(np.float32)
print(f"\nNSER={NSER} NSL={NSL} PX={PX} | NaN targets {int(nan_mask.sum())}")
print("GPU:", torch.cuda.get_device_name(0))

### 2 · Splits and loss weights — same as the 160px runs

In [ ]:
gold_ids = set(gold["StudyInstanceUID"])
is_gold  = train["StudyInstanceUID"].isin(gold_ids).values

t0 = time.time()
present = np.ones((N, NSER), np.float32)
for i in range(N):
    for k in range(2):
        present[i, k] = float(arrA[i, k, NSL // 2].max() > 0)
    present[i, 2] = float(arrB[i, 0, NSL // 2].max() > 0)
print(f"presence in {time.time()-t0:.0f}s | mean series/study {present.sum(1).mean():.2f}")

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    os.system("pip install -q iterative-stratification")
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

usable = np.where((~is_gold) & (present.sum(1) > 0) & (~nan_mask))[0]
Ybin   = (Y[usable] > 0.5).astype(int)
mskf   = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds  = np.zeros(len(usable), int)
for f, (_, v) in enumerate(mskf.split(usable, Ybin)):
    folds[v] = f
gd_idx = np.where(is_gold & (present.sum(1) > 0))[0]
print("fold sizes:", [int((folds == f).sum()) for f in range(5)], "| gold", len(gd_idx))

gl_ids  = gold["StudyInstanceUID"].tolist()
Pg_ext  = np.nan_to_num(pseudo.set_index("StudyInstanceUID").loc[gl_ids, KEYS].values.astype(np.float32), nan=20.0)
Yg_true = gold.set_index("StudyInstanceUID").loc[gl_ids, LABELS].values.astype(int)
ext_auc = np.array([roc_auc_score(Yg_true[:, c], Pg_ext[:, c]) for c in range(12)])
w = ext_auc - 0.5; w = w / w.mean()
loss_w = torch.tensor(w, dtype=torch.float32).cuda()
print("loss weights:", np.round(w, 2))

### 3 · Dataset

Returns `(S, T, H, W)`. The 2.5D channel stack is built on the GPU in `forward`, so the DataLoader carries a third of the data it otherwise would.

In [ ]:
class KneeDS(Dataset):
    def __init__(self, idx, train_mode):
        self.idx, self.train_mode = idx, train_mode
    def __len__(self): return len(self.idx)
    def __getitem__(self, n):
        i = self.idx[n]
        v = np.concatenate([np.asarray(arrA[i]), np.asarray(arrB[i])], 0).astype(np.float32) / 255.0
        if self.train_mode:
            ang   = np.random.uniform(-12, 12)
            scale = np.random.uniform(0.9, 1.1)
            tx    = np.random.uniform(-0.06, 0.06) * PX
            ty    = np.random.uniform(-0.06, 0.06) * PX
            M = cv2.getRotationMatrix2D((PX/2, PX/2), ang, scale)
            M[0, 2] += tx; M[1, 2] += ty
            gain = np.random.uniform(0.85, 1.15)
            bias = np.random.uniform(-0.08, 0.08)
            for s in range(NSER):
                if v[s].max() == 0: continue
                for j in range(NSL):
                    v[s, j] = cv2.warpAffine(v[s, j], M, (PX, PX), borderValue=0)
                v[s] = np.clip(v[s] * gain + bias, 0, 1)
        return torch.from_numpy(v), torch.from_numpy(present[i]), torch.from_numpy(Y[i])

def loader(idx, train_mode, bs):
    return DataLoader(KneeDS(idx, train_mode), batch_size=bs, shuffle=train_mode,
                      num_workers=2, pin_memory=True, drop_last=train_mode,
                      persistent_workers=True)
print("dataset ready")

### 4 · Model — 2.5D built on GPU

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    def __init__(self, backbone="efficientnet_b0", nlab=12, hid=256, pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))

    def forward(self, v, mask):          # v: (B, S, T, H, W)
        B, S, T, H, W = v.shape
        lo = torch.cat([v[:, :, :1],  v[:, :, :-1]], 2)
        hi = torch.cat([v[:, :, 1:],  v[:, :, -1:]], 2)
        x  = torch.stack([lo, v, hi], 3).reshape(B * S * T, 3, H, W)
        f = self.enc(x).reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))

def macro_auc(P, T, thresh=0.5):
    P = np.nan_to_num(P, nan=0.5); out = []
    for c in range(T.shape[1]):
        y = (T[:, c] > thresh).astype(int)
        out.append(np.nan if y.min() == y.max() else roc_auc_score(y, P[:, c]))
    return np.nanmean(out), out

m = Net().cuda()
print("params:", round(sum(p.numel() for p in m.parameters())/1e6, 2), "M")
del m; torch.cuda.empty_cache()

### 5 · Train fold 0

Batch of 3 studies with 2-step gradient accumulation, so the effective batch stays at 6 — matching the 160px runs, which keeps the comparison fair.

In [ ]:
FOLD, EPOCHS, BS, ACCUM, LR = 0, 8, 3, 2, 3e-4

seed_all(42 + FOLD)
tr_idx = usable[folds != FOLD]
va_idx = usable[folds == FOLD]
tl, vl, gl = loader(tr_idx, True, BS), loader(va_idx, False, BS), loader(gd_idx, False, BS)
print(f"train {len(tr_idx):,} | val {len(va_idx):,} | gold {len(gd_idx)}")

model  = Net().cuda()
opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps  = EPOCHS * (len(tl) // ACCUM + 1)
sched  = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")
lossf  = nn.BCEWithLogitsLoss(weight=loss_w)

@torch.no_grad()
def ev(dl):
    model.eval(); P, T = [], []
    for v, mk, y in dl:
        with torch.amp.autocast("cuda"):
            p = model(v.cuda(non_blocking=True), mk.cuda())
        P.append(p.float().sigmoid().cpu().numpy()); T.append(y.numpy())
    return np.concatenate(P), np.concatenate(T)

best = 0
for ep in range(EPOCHS):
    model.train(); tot = 0; t0 = time.time()
    opt.zero_grad(set_to_none=True)
    for bi, (v, mk, y) in enumerate(tl):
        v, mk, y = v.cuda(non_blocking=True), mk.cuda(), y.cuda()
        with torch.amp.autocast("cuda"):
            loss = lossf(model(v, mk), y) / ACCUM
        scaler.scale(loss).backward()
        if (bi + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
        tot += loss.item() * ACCUM
        if not math.isfinite(loss.item()):
            print("non-finite loss, stopping"); break
        if bi % 200 == 0:
            el = time.time() - t0
            print(f"  ep{ep} {bi}/{len(tl)} loss {tot/(bi+1):.4f} eta {(len(tl)-bi)*el/(bi+1)/60:.0f}m", flush=True)

    Pv, Tv = ev(vl); av, _   = macro_auc(Pv, Tv)
    Pg, Tg = ev(gl); ag, per = macro_auc(Pg, Tg)
    print(f"EPOCH {ep}  loss {tot/len(tl):.4f}  val {av:.4f}  gold {ag:.4f}  [{(time.time()-t0)/60:.0f}m]", flush=True)

    if av > best:
        best = av
        torch.save({"model": {k: t.cpu() for k, t in model.state_dict().items()},
                    "fold": FOLD, "val_auc": av, "gold_auc": ag, "epoch": ep,
                    "px": PX, "nsl": NSL}, "/kaggle/working/model224_fold0.pt")
        oof = pd.DataFrame(Pv, columns=LABELS)
        oof.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values[va_idx])
        oof.to_csv("/kaggle/working/oof224_fold0.csv", index=False)
        print("   saved")

print(f"\nbest val {best:.4f}")

### 6 · Did the laterality fix work?

The decisive comparison. At 160px the medial/lateral gaps were **0.098** for meniscus and **0.092** for osteoarthritis. If mirroring is now correct, those should shrink substantially — regardless of what resolution did to the overall score.

In [ ]:
print("per-label, 224px fold 0, gold holdout:")
for l, a in zip(LABELS, per):
    print(f"  {l:<18} {a:.3f}")

d = dict(zip(LABELS, per))
mm, lm = d["Medial Meniscus"], d["Lateral Meniscus"]
mo, lo_ = d["Medial OA"], d["Lateral OA"]
print(f"\n{'':<22}{'medial':>9}{'lateral':>9}{'gap':>9}{'was':>9}")
print(f"{'Meniscus':<22}{mm:>9.3f}{lm:>9.3f}{mm-lm:>+9.3f}{0.098:>9.3f}")
print(f"{'Osteoarthritis':<22}{mo:>9.3f}{lo_:>9.3f}{mo-lo_:>+9.3f}{0.092:>9.3f}")
print("\n(gaps shrinking toward 0 => the mirroring fix landed)")
print(f"\nreference: 160px fold0 gold 0.8033 | 160px 5-fold ens gold 0.825, LB 0.856")